# 02 — Quality Control Contact Sheets

Generate per-slide thumbnail grids and a master contact sheet so you can visually
identify artifacts, debris, or missed tissue sections.

**Prerequisites:**
- Extracted tissue tiles from **notebook 01**
- `pip install -e .` from the `wsi-tissue-pipeline` root

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from wsi_pipeline.qc_grid import build_qc_grids, find_images, group_by_slide

%load_ext autoreload
%autoreload 2

## Step 1: Configure Paths

In [ ]:
# --- Edit for your data ---
img_root_dir = Path(
    "./data/output"  # ← Replace with the path to your extracted tissue tiles (output of notebook 01)
).expanduser().resolve()

qc_output_dir = img_root_dir / "_qc_grids"

print(f"Image root:  {img_root_dir}")
print(f"QC output:   {qc_output_dir}")

## Step 2: Build QC Grids

`build_qc_grids()` discovers images, groups them by slide, generates per-slide
thumbnail grids, and optionally creates a master contact sheet.

Parameters:
- **`thumb_size`** — thumbnail resolution in pixels
- **`columns`** — number of grid columns (or `"auto"`)
- **`label_mode`** — `"both"` shows per-slide and global indices
- **`backend`** — `"auto"` uses torch if available, falls back to PIL

In [ ]:
grid_paths = build_qc_grids(
    img_root_dir,
    qc_output_dir,
    thumb_size=256,
    padding=0,
    columns="auto",
    label_mode="both",
    backend="auto",
    create_master=True,
)

print(f"Generated {len(grid_paths)} grid file(s)")
for p in grid_paths:
    print(f"  {p.name}")

## Step 3: Display the Master Contact Sheet

In [ ]:
master = qc_output_dir / "master_contact_sheet.png"
if master.exists():
    plt.figure(figsize=(12, 12))
    plt.axis("off")
    plt.title("Master Contact Sheet")
    plt.imshow(Image.open(master))
    plt.show()
else:
    print("Master contact sheet not found.")

## Step 4: Display Per-Slide Grids

In [ ]:
per_slide = sorted(qc_output_dir.glob("slide_*.png"))
for grid_path in per_slide:
    plt.figure(figsize=(8, 6))
    plt.axis("off")
    plt.title(grid_path.stem)
    plt.imshow(Image.open(grid_path))
    plt.show()

## Step 5: Identify and Remove Artifacts

Look at the contact sheet above, note the label of any non-tissue tile (glass,
debris), and delete it from the output directory.  After deletion, re-run
`rename_outputs_by_overall_index()` in **notebook 01, step 5** to fix the
global index sequence.

In [ ]:
# Example: delete a problematic tile identified from the contact sheet
fname_to_delete = img_root_dir / "EXAMPLE_NAME.tif"
if fname_to_delete.exists():
    fname_to_delete.unlink()
    print(f"Deleted: {fname_to_delete.name}")
else:
    print(f"Not found (expected): {fname_to_delete.name}")

## Step 6: Regenerate QC After Cleanup

After deleting artifacts and re-running the renaming step, regenerate the
contact sheets to confirm everything looks clean.

In [ ]:
# Re-run the same build_qc_grids call as Step 2
grid_paths = build_qc_grids(
    img_root_dir,
    qc_output_dir,
    thumb_size=256,
    padding=0,
    columns="auto",
    label_mode="both",
    backend="auto",
    create_master=True,
)
print(f"Regenerated {len(grid_paths)} grid file(s)")

## Optional: Image Statistics

Compute per-image dimensions and intensity statistics, then save to CSV for
further analysis or anomaly detection.

In [ ]:
import pandas as pd

image_paths = find_images(img_root_dir)
rows = []
for p in image_paths:
    img = Image.open(p)
    arr = np.array(img)
    rows.append({
        "filename": p.name,
        "width": img.width,
        "height": img.height,
        "area_px": img.width * img.height,
        "mean_intensity": float(arr.mean()),
        "std_intensity": float(arr.std()),
    })

df = pd.DataFrame(rows)
print(df.describe())

csv_path = qc_output_dir / "image_statistics.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved to {csv_path}")

## Next Steps

- **Notebook 03** — Visualize extracted tissues in Neuroglancer
- **Notebook 04** — Prepare the image stack for EM-LDDMM registration